<div style="display: table; width: 100%;">
  <div style="display: table-cell; text-align: center; vertical-align: middle; width: 70%;">
    <h1>INTELIGENCIA ARTIFICAL: DATA MINING II</h1>
  </div>
  <div style="display: table-cell; text-align: center; vertical-align: middle; width: 30%;">
    <img src="https://github.com/UIDE-Tareas/9-Inteligencia-Artificial-Data-Mining-II-Tarea1/blob/main/Assets/UideLogo.png?raw=true" alt="logo UIDE" style="width:50%;">
  </div>
</div>
<hr />

### 🟦 Componente Práctico 1  
🟡 Grupo: 5      
🟡 Semana: 1      
🟡 Docente:  MIGUEL ÁNGEL ROSERO AGUAS    

### 🟦 Realizado por:   
Estudiantes

💻 EVELIN MARISOL ROSERO ORDOÑEZ   

💻 MAYRA CECILIA SALAZAR GRANDES

💻 SAMANTHA CAROLINA BUITRON PAMBABAY

💻 JOSE MANUEL ESPINOZA BONE

### 🟦 Objetivo y alcance del trabajo 
El objetivo de esta actividad es aplicar conocimientos en paquetes de análisis de datos en Python, específicamente utilizando NumPy, Pandas y SKLearn, para realizar un análisis exhaustivo del conjunto de datos "Social Media Usage and Emotional Well-Being" disponible en Kaggle (https://www.kaggle.com/datasets/emirhanai/social-media-usage-and-emotional-well-being/data). Los estudiantes deberán descargar e importar los datos, transformarlos en dataframes de Pandas y llevar a cabo un análisis exploratorio de datos. Esto incluirá la creación de histogramas de distribución para cada característica y la obtención de descriptores estadísticos.


Además, se realizará un análisis de regresión para estudiar las relaciones entre "Daily_Usage_Time" y "Likes_Received_Per_Day", "Daily_Usage_Time" y "Posts_Per_Day", así como entre "Posts_Per_Day" y "Likes_Received_Per_Day". Las y los estudiantes también llevarán a cabo un análisis de clustering utilizando todas las características disponibles, seguido de una selección de las variables más importantes. Se realizará un análisis de reducción de dimensionalidad para simplificar el modelo.


Finalmente, los estudiantes deberán realizar una clasificación supervisada utilizando "Dominant_Emotion" como etiqueta objetivo. Se probarán diferentes métodos de clasificación y se evaluará el mejor modelo en un conjunto de prueba. 


El reto que se presenta a las y los estudiantes es integrar y aplicar diversas técnicas de análisis de datos y machine learning para obtener información valiosa a partir del conjunto de datos. Este trabajo permitirá a las y los estudiantes entender cómo el uso de redes sociales puede estar relacionado con el bienestar emocional, utilizando herramientas y técnicas avanzadas de análisis de datos.

### 🟦 [Código fuente original](https://github.com/UIDE-Tareas/9-Inteligencia-Artificial-Data-Mining-II-Tarea1.git)
Con [git](https://git-scm.com/) instalado. En Windows, Linux o MacOS ejecutar el comando.

```
git clone "https://github.com/UIDE-Tareas/9-Inteligencia-Artificial-Data-Mining-II-Tarea1.git"
```

<hr />

# 0️⃣ Preparar entorno

Funciones base para utilizar si son requeridas en el presente notebook. Adicional hay funciones utilitarias para utilizar con pandas.DataFrame y finalmente las funciones para cumplir con los objetivos del presente trabajo práctico.

In [ ]:
# UTILIDADES PARA GESTIÓN DE DEPENDENCIAS, INFORMACIÓN DEL ENTORNO Y FUNCIONES AUXILIARES

import sys
import subprocess
import os
from pathlib import Path
from enum import Enum
import zipfile
from typing import Optional
from typing import Iterable
from dataclasses import dataclass
from typing import cast
from typing import Tuple
from enum import Enum
from types import SimpleNamespace
from typing import Any
from typing import Protocol
from typing import Literal
from typing import Sequence

# Libs a instalar
LIBS = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "requests",
    "wcwidth",
]

class ConsoleColor(Enum):
    RED = "\033[91m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    BLUE = "\033[94m"
    MAGENTA = "\033[95m"
    CYAN = "\033[96m"
    WHITE = "\033[97m"
    RESET = "\033[0m"


def PrintColor(message: str, color: ConsoleColor) -> str:
    RESET = ConsoleColor.RESET.value
    return f"{color.value}{message}{RESET}"


def ShowMessage(
    message: str, title: str, icon: str, color: ConsoleColor, end: str = "\n"
):
    colored_title = PrintColor(icon + f"  " + title.upper() + ":", color)
    print(f"{colored_title} {message}", end=end)


def ShowInfoMessage(
    message: str, title: str = "Info", icon: str = "ℹ️", end: str = "\n"
):
    ShowMessage(message, title, icon, ConsoleColor.CYAN, end)


def ShowSuccessMessage(
    message: str, title: str = "Success", icon: str = "✅", end: str = "\n"
):
    ShowMessage(message, title, icon, ConsoleColor.GREEN, end)


def ShowErrorMessage(
    message: str, title: str = "Error", icon: str = "❌", end: str = "\n"
):
    ShowMessage(message, title, icon, ConsoleColor.RED, end)


def ShowWarningMessage(
    message: str, title: str = "Warning", icon: str = "⚠️", end: str = "\n"
):
    ShowMessage(message, title, icon, ConsoleColor.YELLOW, end)


# Funcion para ejecutar comandos
def RunCommand(
    commandList: list[str], printCommand: bool = True, printError: bool = True
) -> subprocess.CompletedProcess[str]:
    print("⏳", " ".join(commandList))

    if printCommand:
        proc = subprocess.Popen(
            commandList,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
            universal_newlines=True,
        )

        out_lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            out_lines.append(line)

        proc.wait()
        err_text = ""
        if proc.stderr is not None:
            err_text = proc.stderr.read() or ""

        if proc.returncode != 0 and printError and err_text:
            ShowErrorMessage(err_text, "", end="")
            # print(err_text, end="")

        return subprocess.CompletedProcess(
            args=commandList,
            returncode=proc.returncode,
            stdout="".join(out_lines),
            stderr=err_text,
        )

    else:
        result = subprocess.run(
            commandList, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        if result.returncode != 0 and printError and result.stderr:
            ShowErrorMessage(result.stderr, "", end="")
            # print(result.stderr, end="")
        return result


# Función para instalar las dependencias
def InstallDeps(libs: Optional[list[str]] = None):
    print("ℹ️ Installing deps.")
    printCommand = False
    printError = True
    RunCommand(
        [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
        printCommand=printCommand,
        printError=printError,
    )
    if libs is None or libs.count == 0:
        print("No hay elementos a instalar.")
    else:
        RunCommand(
            [sys.executable, "-m", "pip", "install", *libs],
            printCommand=printCommand,
            printError=printError,
        )
        print("Deps installed.")
    print()


# Función para mostrar info el ambiente de ejecución
def ShowEnvironmentInfo():
    print("ℹ️  Environment Info:")
    print("Python Version:", sys.version)
    print("Platform:", sys.platform)
    print("Executable Path:", sys.executable)
    print("Current Working Directory:", os.getcwd())
    print("VIRTUAL_ENV:", os.environ.get("VIRTUAL_ENV"))
    print("sys.prefix:", sys.prefix)
    print("sys.base_prefix:", sys.base_prefix)
    print()


InstallDeps(LIBS)
ShowEnvironmentInfo()
import requests


@dataclass(frozen=True)
class BoxStyle:
    TL: str
    TR: str
    BL: str
    BR: str
    H: str
    V: str

class TitleBoxLineStyle(Enum):
    SIMPLE = BoxStyle("┌", "┐", "└", "┘", "─", "│")
    DOUBLE = BoxStyle("╔", "╗", "╚", "╝", "═", "║")
    ROUNDED = BoxStyle("╭", "╮", "╰", "╯", "─", "│")
    HEAVY = BoxStyle("┏", "┓", "┗", "┛", "━", "┃")
    ASCII = BoxStyle("+", "+", "+", "+", "-", "|")
    DOUBLE_BOLD = BoxStyle("╔", "╗", "╚", "╝", "╬", "║")
    BLOCK = BoxStyle("█", "█", "█", "█", "█", "█")
    HEAVY_CROSS = BoxStyle("╒", "╕", "╘", "╛", "╪", "┃")
    METAL = BoxStyle("╞", "╡", "╘", "╛", "═", "║")


# Función para mostrar un título con recuadro
def ShowTitleBox(
    text: str,
    max_len: int = 100,
    boxLineStyle: TitleBoxLineStyle = TitleBoxLineStyle.SIMPLE,
    color: ConsoleColor = ConsoleColor.CYAN,
):
    try:

        def vislen(s: str) -> int:
            from wcwidth import wcswidth as _w

            n = _w(s)
            return n if n >= 0 else len(s)

    except Exception:

        def vislen(s: str) -> int:
            return len(s)

    pad = 1
    tlen = vislen(text)
    inner = max(max_len, tlen)
    left = (inner - tlen) // 2
    right = inner - tlen - left

    top = f"{boxLineStyle.value.TL}{boxLineStyle.value.H * (inner + 2 * pad)}{boxLineStyle.value.TR}"
    mid = f"{boxLineStyle.value.V}{' ' * pad}{' ' * left}{text}{' ' * right}{' ' * pad}{boxLineStyle.value.V}"
    bot = f"{boxLineStyle.value.BL}{boxLineStyle.value.H * (inner + 2 * pad)}{boxLineStyle.value.BR}"
    print(PrintColor("\n".join([top, mid, bot]), color))


# Función para descargar un archivo
def DownloadFile(uri: str, filename: str, overwrite: bool = False, timeout: int = 20, printInfo: bool = True):
    dest = Path(filename).resolve()
    if dest.exists() and dest.is_file() and dest.stat().st_size > 0 and not overwrite:
        if printInfo:
            print(
                f'✅ Ya existe: "{dest}". No se descarga (use overwrite=True para forzar).'
            )
        return
    if dest.parent and not dest.parent.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
    if printInfo:
        print(f'ℹ️ Descargando "{uri}" → "{dest}"')
    try:
        with requests.get(uri, stream=True, timeout=timeout) as resp:
            resp.raise_for_status()
            tmp = dest.with_suffix(dest.suffix + ".part")
            with open(tmp, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 64):
                    if chunk:  # filtra keep-alive chunks
                        f.write(chunk)
            tmp.replace(dest)
        if printInfo: 
            print(f'✅ Archivo "{dest}" descargado exitosamente.')
    except requests.exceptions.RequestException as e:
        print(f"❌ Error al descargar: {e}")


# Función para descomprimir un archivo zip
def UnzipFile(filename: str, outputDir: str):
    print(f'ℹ️ Descomprimiendo "{filename}" en "{outputDir}"')
    try:
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall(outputDir)
        print(f"Descomprimido en: {os.path.abspath(outputDir)}")
    except Exception as e:
        print(f"Error: {e}")


In [ ]:
# UTILIDADES PARA ANÁLISIS Y MANIPULACIÓN DE DATAFRAMES

# Importar libraries
import random
import pandas as pd
import pandas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from matplotlib.figure import Figure
from matplotlib.axes import Axes

from pandas import DataFrame
from pandas import Series
from sklearn.utils import Bunch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# Configurar opciones de Pandas
pd.set_option("display.float_format", "{:.2f}".format)
pandas.set_option("display.max_rows", None)
pandas.set_option("display.max_columns", None)


# Función para mostrar la información del DataFrame.
def ShowDfInfo(df: pandas.DataFrame, title):
    display(f"ℹ️ INFO {title} ℹ️")
    df.info()
    display()


# Función para mostrar las n primeras filas del DataFrame.
def ShowDfHead(df: pandas.DataFrame, title: str, headQty=10):
    display(f"ℹ️ {title}: Primeros {headQty} elementos.")
    display(df.head(headQty))
    display()


# Función para mostrar las n últimas filas del DataFrame.
def ShowDfTail(df: pandas.DataFrame, title: str, tailQty=10):
    display(f"ℹ️ {title}: Últimos {tailQty} elementos.")
    display(df.tail(tailQty))
    display()


# Mostrar el tamaño del DataFrame
def ShowDfShape(df: pandas.DataFrame, title: str):
    display(f"ℹ️ {title} - Tamaño de los datos")
    display(f"{df.shape[0]} filas x {df.shape[1]} columnas")
    display()


# Función para mostrar la estadística descriptiva de todas las columnas del DataFrame, por tipo de dato.
def ShowDfStats(df: pandas.DataFrame, title: str = ""):
    display(f"ℹ️ Estadística descriptiva - {title}")
    numeric_cols = df.select_dtypes(include="number")
    if not numeric_cols.empty:
        display("    🔢 Columnas numéricas".upper())
        numeric_desc = (
            numeric_cols.describe().round(2).T
        )  # Transpuesta para añadir columna
        numeric_desc["var"] = numeric_cols.var(numeric_only=True).round(2)
        display(numeric_desc.T)
    non_numeric_cols = df.select_dtypes(
        include=["boolean", "string", "category", "object"]
    )
    if not non_numeric_cols.empty:
        display("    🔡 Columnas no numéricas".upper())
        non_numeric_desc = non_numeric_cols.describe()
        display(non_numeric_desc)
    datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"])
    if not datetime_cols.empty:
        display("    📅 Columnas fechas".upper())
        datetime_desc = datetime_cols.describe()
        display(datetime_desc)

# Función para mostrar los duplicados de un DataFrame, agrupados por todas las columnas y ordenados por cantidad de ocurrencias.
def ShowDfDuplicates(
    df: pandas.DataFrame,
    title: str,
    qty: int = 10
):
    display(f"ℹ️ Duplicados en el DataFrame - {title}")

    dup_mask = df.duplicated()
    dup_df = df[dup_mask]
    dup_count = dup_df.shape[0]

    display(f"Cantidad de filas duplicadas: {dup_count}")

    if dup_count == 0:
        display("No se encontraron duplicados.")
        display()
        return

    display(f"Mostrando hasta {qty} duplicados:")
    display(dup_df.head(qty))

    remaining = dup_count - qty
    if remaining > 0:
        display(f"⚠️ Existen {remaining} filas duplicadas adicionales no mostradas.")

    display()

# Función para mostrar los valores únicos de cada columna, con un límite opcional para no mostrar demasiados valores.
def ShowDfUniqueValues(df: pandas.DataFrame, title: str, qty: int = 30):
    display(f"ℹ️ Valores únicos por columna - {title}")

    for col in df.columns:
        unique_vals = df[col].dropna().unique()
        count = len(unique_vals)

        display(f"Columna: {col}")
        display(f"Cantidad de valores únicos: {count}")

        if count <= qty:
            display(sorted(unique_vals))
        else:
            display(f"Más de {qty} valores únicos. Mostrando primeros {qty}:")
            display(sorted(unique_vals[:qty]))

        display()

# Función para mostrar una visión general completa del DataFrame
def ShowFullDfOverview(df, title, headQty=5, tailQty=5, duplicatesqty = 10, uniqueqty = 30):
    ShowDfInfo(df, title)
    ShowDfStats(df, title)
    ShowDfShape(df, title)
    ShowDfDuplicates(df, title, qty=duplicatesqty)
    ShowDfUniqueValues(df, title, qty=uniqueqty)
    ShowDfHead(df, title, headQty=headQty)
    ShowDfTail(df, title, tailQty=tailQty)


# Función para mostrar los valores nulos o NaN de cada columna en un DataFrame
def ShowDfNanValues(df: pandas.DataFrame, title: str):
    display(f"ℹ️ Contador de valores Nulos - {title}")
    nulls_count = df.isnull().sum()
    nulls_df = nulls_count.reset_index()
    nulls_df.columns = ["Columna", "Cantidad_Nulos"]
    display(nulls_df)
    display()


# Tipos de correlación
class CorrelationType(Enum):
    ALL = "all"
    STRONG = "strong"
    WEAK = "weak"


# Muestra las correlaciones completas, débiles y fuertes.
def ShowDfCorrelation(
    df: pandas.DataFrame,
    title: str,
    fig: Optional[Figure] = None,
    ax: Optional[Axes] = None,
    level: CorrelationType = CorrelationType.ALL,
    umbral: float = 0.6,
    showTable: bool = False,
    figsize: tuple = (8, 6),
    annotate: bool = True,
):
    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=figsize)

    display(f"ℹ️ {title.upper()} - MATRIZ DE CORRELACIÓN ({level.name})")

    corr = df.select_dtypes(include="number").corr()

    if level == CorrelationType.STRONG:
        corr = corr.where(np.abs(corr) >= umbral)
    elif level == CorrelationType.WEAK:
        corr = corr.where((np.abs(corr) < umbral) | (corr == 1))
    elif level != CorrelationType.ALL:
        raise ValueError(f"Invalid level: {level}")

    # ✅ Mostrar diagonal (1) y triángulo inferior
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    sns.heatmap(
        corr,
        mask=mask,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        annot=annotate,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "Coeficiente de correlación"},
        ax=ax
    )

    subtitle = (
        "Completa"
        if level == CorrelationType.ALL
        else f"Strong (|r| ≥ {umbral})"
        if level == CorrelationType.STRONG
        else f"Weak (|r| < {umbral})"
    )

    ax.set_title(
        f"Matriz de correlación ({subtitle})",
        fontsize=12,
        pad=15
    )

    ax.tick_params(axis="x", rotation=90)
    ax.tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.show()

    if showTable:
        display(corr.round(3))

    return fig, corr

# Función para normalizar los nombres de las columnas de un DataFrame (strip, title, remove spaces and underscores).
def NormalizeColumnNames(df: pandas.DataFrame) -> pandas.DataFrame:
    df.columns = [
        col.strip().title().replace(" ", "").replace("_", "") for col in df.columns
    ]
    return df

# Función para eliminar columnas de un DataFrame, si existen, con opción inplace o retornando una copia.
def DropColumns(
    df: pandas.DataFrame, toDrop: list[str], inplace: bool = False
) -> pandas.DataFrame:
    if not toDrop:
        return df
    if inplace:
        df.drop(columns=df.columns.intersection(toDrop), inplace=True)
        return df
    else:
        return df.drop(columns=df.columns.intersection(toDrop))


# Para almacenar los datos del dataset
@dataclass
class Dataset:
    X: pandas.DataFrame
    y: pandas.DataFrame


# Para almacenar los datos de split del dataset.
@dataclass
class DatasetSplit:
    Train: Dataset
    Test: Dataset


# Muestra el head de cada componente del split.
def ShowDatasetSplitHead(split: DatasetSplit, title: str, headQty: int = 5):
    ShowDfHead(split.Train.X, f"{title} - X Train", headQty)
    ShowDfHead(split.Train.y, f"{title} - y Train", headQty)
    ShowDfHead(split.Test.X, f"{title} - X Test", headQty)
    ShowDfHead(split.Test.y, f"{title} - y Test", headQty)


# Muestra la información del Dataset
def ShowDatasetInfo(data: Dataset, title):
    tAux = title
    title = f"{tAux} - Caracteristicas - X"
    ShowDfInfo(data.X, title)
    ShowDfShape(data.X, title)
    ShowDfStats(data.X, title)
    ShowDfNanValues(data.X, title)
    ShowDfHead(data.X, title)
    ShowDfTail(data.X, title)
    title = f"{tAux} - Características - y"
    ShowDfInfo(data.y, title)
    ShowDfShape(data.y, title)
    ShowDfStats(data.y, title)
    ShowDfNanValues(data.y, title)
    ShowDfHead(data.y, title)
    ShowDfTail(data.y, title)


# Muestra la información del Dataset Split
def ShowDatasetSplitInfo(split: DatasetSplit, title: str, headQty: int = 5):
    tAux = title
    title = f"{tAux} - TRAIN"
    ShowDatasetInfo(split.Train, title)
    title = f"{tAux} - TEST"
    ShowDatasetInfo(split.Test, title)


# Realiza el split del Dataset, en Train y test utilizando el ratio.
def SplitDataset(
    data: Dataset, trainRatio: float = 0.8, randomState: int = 42
) -> DatasetSplit:
    y_strat = data.y.iloc[:, 0]
    XTrain, XTest, yTrain, yTest = train_test_split(
        data.X,
        data.y,
        train_size=trainRatio,
        random_state=randomState,
        stratify=y_strat,
    )
    return DatasetSplit(
        Train=Dataset(X=XTrain.reset_index(drop=True), y=yTrain.reset_index(drop=True)),
        Test=Dataset(X=XTest.reset_index(drop=True), y=yTest.reset_index(drop=True)),
    )


# Contrato para los escaladores
class ScalerProtocol(Protocol):
    def fit(self, X, y: Any = None) -> Any: ...
    def transform(self, X) -> Any: ...
    def fit_transform(self, X, y: Any = None) -> Any: ...


# Para almacenar los datos del dataset aplicado el escalador.
@dataclass
class ScaledDatasetSplit(DatasetSplit):
    Scaler: ScalerProtocol

# Enum para los tipos de escaladores soportados
class ScalerType(Enum):
    STANDARD = "Standard"
    MIN_MAX = "minmax"
    ROBUST = "robust"
    MAX_ABS = "maxabs"
    NORMALIZER = "normalizer"
    QUANTILE = "quantile"
    POWER = "power"
    FUNCTION = "function"


# Crea una instancia de scaler según el Enum ScalerType.
def CreateScaler(scalerType: ScalerType, **kwargs) -> ScalerProtocol:
    if scalerType == ScalerType.STANDARD:
        return StandardScaler(**kwargs)
    if scalerType == ScalerType.MIN_MAX:
        return MinMaxScaler(**kwargs)
    if scalerType == ScalerType.ROBUST:
        return RobustScaler(**kwargs)
    if scalerType == ScalerType.MAX_ABS:
        return MaxAbsScaler(**kwargs)
    if scalerType == ScalerType.NORMALIZER:
        return Normalizer(**kwargs)
    if scalerType == ScalerType.QUANTILE:
        return QuantileTransformer(**kwargs)
    if scalerType == ScalerType.POWER:
        return PowerTransformer(**kwargs)
    if scalerType == ScalerType.FUNCTION:
        return FunctionTransformer(**kwargs)
    raise ValueError(f"ScalerType no soportado: {scalerType}")

def DetectScaler(scaler: ScalerProtocol) -> ScalerType:
    if isinstance(scaler, StandardScaler):
        return ScalerType.STANDARD
    if isinstance(scaler, MinMaxScaler):
        return ScalerType.MIN_MAX
    if isinstance(scaler, RobustScaler):
        return ScalerType.ROBUST
    if isinstance(scaler, MaxAbsScaler):
        return ScalerType.MAX_ABS
    if isinstance(scaler, Normalizer):
        return ScalerType.NORMALIZER
    if isinstance(scaler, QuantileTransformer):
        return ScalerType.QUANTILE
    if isinstance(scaler, PowerTransformer):
        return ScalerType.POWER
    if isinstance(scaler, FunctionTransformer):
        return ScalerType.FUNCTION
    raise ValueError(f"No se reconoce el tipo de scaler: {type(scaler)}")

# Escala el split usando el escalador proporcionado y retorna el split escalado.
def ScaleDatasetSplit(
    split: DatasetSplit, scaler: ScalerProtocol = StandardScaler()
) -> ScaledDatasetSplit:
    XTrainScaledValues = scaler.fit_transform(split.Train.X)
    XTestScaledValues = scaler.transform(split.Test.X)

    XTrainScaled = pandas.DataFrame(
        XTrainScaledValues, columns=split.Train.X.columns, index=split.Train.X.index
    )

    XTestScaled = pandas.DataFrame(
        XTestScaledValues, columns=split.Test.X.columns, index=split.Test.X.index
    )

    TrainScaledDataset = Dataset(X=XTrainScaled, y=split.Train.y.copy())
    TestScaledDataset = Dataset(X=XTestScaled, y=split.Test.y.copy())

    return ScaledDatasetSplit(
        Train=TrainScaledDataset, Test=TestScaledDataset, Scaler=scaler
    )

# Para almacenar los datos del dataset aplicado PCA.
@dataclass
class PcaDatasetSplit(DatasetSplit):
    Pca: PCA
    Scaler: ScalerProtocol | None = None 

# Aplica PCA al split escalado y retorna el split con PCA aplicado.
def ApplyPCA(
    split: ScaledDatasetSplit,
    explainedVarianceRatioSum: float = 0.95,
    randomState: int = 42
) -> PcaDatasetSplit:

    def GetPCNames(n: int) -> list[str]:
        return [f"PC{i}" for i in range(1, n + 1)]

    pca = PCA(n_components=explainedVarianceRatioSum, random_state=randomState)

    XTrainPCA = pca.fit_transform(split.Train.X)
    XTestPCA = pca.transform(split.Test.X)

    XTrainPcaDf = pandas.DataFrame(
        XTrainPCA, index=split.Train.X.index, columns=GetPCNames(XTrainPCA.shape[1])
    )

    XTestPcaDf = pandas.DataFrame(
        XTestPCA, index=split.Test.X.index, columns=GetPCNames(XTestPCA.shape[1])
    )

    return PcaDatasetSplit(
        Train=Dataset(X=XTrainPcaDf, y=split.Train.y.copy()),
        Test=Dataset(X=XTestPcaDf, y=split.Test.y.copy()),
        Pca=pca,
        Scaler=split.Scaler
    )

# Tipo de split que puede ser escalado o con PCA aplicado.
SplitLike = ScaledDatasetSplit | PcaDatasetSplit


from sklearn.metrics import roc_auc_score



# Función para graficar la matriz de confusión
def PlotConfusionMatrix(
    cm,
    classNames: Sequence[str] | None = None,
    title: str = "Confusion Matrix"
):
    # Si classNames es None → usar "auto"
    xticks = classNames if classNames is not None else "auto"
    yticks = classNames if classNames is not None else "auto"

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=xticks,
        yticklabels=yticks
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Utilidades para detección de tipos de split
@dataclass(frozen=True)
class SplitTypeInfo:
    IsPCA: bool
    IsScaled: bool
    IsRaw: bool

# Detecta el tipo de split (PCA, Escalado, Crudo)
def DetectSplitType(split) -> SplitTypeInfo:
    isPca = isinstance(split, PcaDatasetSplit)
    isScaled = isinstance(split, ScaledDatasetSplit)
    isRaw = not isPca and not isScaled

    return SplitTypeInfo(
        IsPCA=isPca,
        IsScaled=isScaled,
        IsRaw=isRaw
    )


# Función para eliminar filas duplicadas de un DataFrame
def RemoveDfDuplicates(df: pandas.DataFrame, inplace: bool = False) -> pandas.DataFrame:
    if inplace:
        df.drop_duplicates(inplace=True)
        return df
    else:
        return df.drop_duplicates()


# 1️⃣ Descarga del Conjunto de Datos
Descarga del archivo comprimido que contiene el dataset y descompresión en un directorio temporal. 

In [ ]:
TEMP_DIR = "Temp"
DATASET_URL = "https://raw.githubusercontent.com/UIDE-Tareas/9-Inteligencia-Artificial-Data-Mining-II-Tarea1/refs/heads/main/Data/archive.zip"
DATASET_FILENAME = os.path.join(TEMP_DIR, "archive.zip")

TEST_FILENAME = os.path.join(TEMP_DIR, "test.csv")
TRAIN_FILENAME = os.path.join(TEMP_DIR, "train.csv")
VAL_FILENAME = os.path.join(TEMP_DIR, "val.csv")

DownloadFile(DATASET_URL, DATASET_FILENAME, overwrite=False)
UnzipFile(DATASET_FILENAME, TEMP_DIR)


# 2️⃣ Importación y Transformación de Datos

In [ ]:
dfTrainOriginal = pandas.read_csv(TRAIN_FILENAME, on_bad_lines="skip")
dfTestOriginal = pandas.read_csv(TEST_FILENAME, on_bad_lines="skip")
dfValOriginal = pandas.read_csv(VAL_FILENAME, on_bad_lines="skip")

# 3️⃣ Análisis Exploratorio de Datos

In [ ]:
# Función para corregir filas donde las columnas de edad y género están intercambiadas o contienen datos inválidos
def FixAgeGenderColumns(
    df: pandas.DataFrame,
    title: str,
    ageCol: str = "Age",
    genderCol: str = "Gender",
    showInvalid: bool = True
) -> pandas.DataFrame:

    def is_number(x):
        return str(x).strip().isdigit()

    df_fixed = df.copy()
    rows_to_drop = []

    for i in df_fixed.index:

        age_val = df_fixed.at[i, ageCol]
        gender_val = df_fixed.at[i, genderCol]

        age_is_num = is_number(age_val)
        gender_is_num = is_number(gender_val)

        if age_is_num and not gender_is_num:
            continue

        elif not age_is_num and gender_is_num:
            df_fixed.at[i, ageCol] = gender_val
            df_fixed.at[i, genderCol] = age_val

        else:
            rows_to_drop.append(i)

    if rows_to_drop and showInvalid:
        display(f"ℹ️ Limpieza de columnas Edad/Género - {title}")
        display(f"⚠️ Filas inválidas detectadas {len(rows_to_drop)} (serán eliminadas):")
        display(df_fixed.loc[rows_to_drop])

    df_fixed.drop(index=rows_to_drop, inplace=True)
    df_fixed.reset_index(drop=True, inplace=True)

    display()

    return df_fixed

# Función para eliminar filas con valores nulos
def RemoveRowsWithNan(df: pandas.DataFrame) -> pandas.DataFrame:
    return df.dropna()

# Función para aplicar tipos de datos
def ApplyDfDtypes(
    df: pandas.DataFrame,
    dtypeMap: dict
) -> pandas.DataFrame:

    df_fixed = df.copy()

    for col, dtype in dtypeMap.items():
        if col in df_fixed.columns:
            df_fixed[col] = df_fixed[col].astype(dtype)

    return df_fixed

# Función para aplicar One-Hot Encoding a múltiples DataFrames garantizando mismas columnas
def ApplyOneHotEncodingMultiDF(
    dfs: list[pandas.DataFrame],
    columns: list[str]
) -> list[pandas.DataFrame]:

    # Guardar tamaños de cada DataFrame
    sizes = [len(df) for df in dfs]

    # Concatenar temporalmente todos los DataFrames
    df_all = pandas.concat(dfs, axis=0, ignore_index=True)

    # Aplicar One-Hot Encoding
    df_all = pandas.get_dummies(
        df_all,
        columns=columns,
        dtype=int
    )

    # Separar nuevamente los DataFrames respetando tamaños originales
    result = []
    start = 0

    for size in sizes:
        result.append(df_all.iloc[start:start+size].reset_index(drop=True))
        start += size

    return result

# Función para mostrar histogramas de columnas numéricas respetando el orden de salida
def ShowDfHistograms(
    df: pandas.DataFrame,
    title: str,
    bins: int = 20,
    figsize: tuple = (12, 8)
):

    # Mostrar primero el título
    display(f"ℹ️ Histogramas de variables numéricas - {title}")

    numeric_cols = df.select_dtypes(include="number")

    fig = numeric_cols.hist(
        figsize=figsize,
        bins=bins
    )

    plt.suptitle(f"Distribución de Variables Numéricas - {title}")

    plt.tight_layout()

    # Forzar render en orden
    plt.show()

    display()


# Primera inspección de los datos
ShowTitleBox("Dataframes iniciales", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)
ShowFullDfOverview(dfTrainOriginal, "Train Original", headQty=10, tailQty=10)
ShowFullDfOverview(dfTestOriginal, "Test Original", headQty=10, tailQty=10)
ShowFullDfOverview(dfValOriginal, "Validation Original", headQty=10, tailQty=10)

# Eliminación de duplicados
dfTrainOriginal = RemoveDfDuplicates(dfTrainOriginal)
dfTestOriginal = RemoveDfDuplicates(dfTestOriginal)
dfValOriginal = RemoveDfDuplicates(dfValOriginal)

# Corrección de columnas Age / Gender
dfTrainOriginal = FixAgeGenderColumns(dfTrainOriginal, "Train Original", showInvalid=False)
dfTestOriginal = FixAgeGenderColumns(dfTestOriginal, "Test Original", showInvalid=False)
dfValOriginal = FixAgeGenderColumns(dfValOriginal, "Validation Original", showInvalid=False)

# Eliminación de NaN
dfTrainOriginal = RemoveRowsWithNan(dfTrainOriginal)
dfTestOriginal = RemoveRowsWithNan(dfTestOriginal)
dfValOriginal = RemoveRowsWithNan(dfValOriginal)

# Tipos de datos
DTYPE = {
    "Age": int,
    "Gender": "string",
    "Platform": "string",
    "Daily_Usage_Time (minutes)": float,
    "Posts_Per_Day": float,
    "Likes_Received_Per_Day": float,
    "Comments_Received_Per_Day": float,
    "Messages_Sent_Per_Day": float,
    "Dominant_Emotion": "string"
}

# Aplicar tipos
dfTrainOriginal = ApplyDfDtypes(dfTrainOriginal, DTYPE)
dfTestOriginal = ApplyDfDtypes(dfTestOriginal, DTYPE)
dfValOriginal = ApplyDfDtypes(dfValOriginal, DTYPE)


# Eliminar columna ID
dfTrainOriginal = dfTrainOriginal.drop(columns=["User_ID"])
dfTestOriginal = dfTestOriginal.drop(columns=["User_ID"])
dfValOriginal = dfValOriginal.drop(columns=["User_ID"])


# Aplicar One-Hot Encoding a Gender,Platform en Train, Test y Validation
dfTrainOriginal, dfTestOriginal, dfValOriginal = ApplyOneHotEncodingMultiDF(
    [dfTrainOriginal, dfTestOriginal, dfValOriginal],
    ["Gender", "Platform"]
)

ShowTitleBox("Histogramas", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)
ShowDfHistograms(dfTrainOriginal, "Train Original - Post Limpieza")


# Copias para clasificación
dfTrainForClassification = dfTrainOriginal.copy()
dfTestForClassification = dfTestOriginal.copy()
dfValForClassification = dfValOriginal.copy()


# Copias para clustering
dfTrainForClustering = dfTrainOriginal.copy()
dfTestForClustering = dfTestOriginal.copy()
dfValForClustering = dfValOriginal.copy()


# Eliminar la emoción para clustering
dfTrainForClustering = dfTrainForClustering.drop(columns=["Dominant_Emotion"])
dfTestForClustering = dfTestForClustering.drop(columns=["Dominant_Emotion"])
dfValForClustering = dfValForClustering.drop(columns=["Dominant_Emotion"])


# Mostrar información final para Clasificación
ShowTitleBox("DataFrames finales para Clasificación", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)
ShowDfInfo(dfTrainForClassification, "Train Clasificación - Limpio")
ShowDfInfo(dfTestForClassification, "Test Clasificación - Limpio")
ShowDfInfo(dfValForClassification, "Validation Clasificación - Limpio")

# Mostrar información final para Clustering
ShowTitleBox("DataFrames finales para Clustering", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)
ShowDfInfo(dfTrainForClustering, "Train Clustering - Limpio")
ShowDfInfo(dfTestForClustering, "Test Clustering - Limpio")
ShowDfInfo(dfValForClustering, "Validation Clustering - Limpio")

# 4️⃣ Análisis de Regresión

In [ ]:
# Función para mostrar regresión lineal con gráfico profesional y métricas
def ShowRegressionPlot(
    df: pandas.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    figsize: tuple = (8,6)
):

    X = df[[x_col]]
    y = df[y_col]

    model = LinearRegression()
    model.fit(X, y)

    coef = model.coef_[0]
    intercept = model.intercept_
    r2 = model.score(X, y)

    display(f"ℹ️ Análisis de Regresión - {title}")
    display(f"Coeficiente (pendiente): {coef:.4f}")
    display(f"Intercepto: {intercept:.4f}")
    display(f"R²: {r2:.4f}")

    plt.figure(figsize=figsize)

    sns.regplot(
        data=df,
        x=x_col,
        y=y_col,
        scatter_kws={"alpha":0.6},
        line_kws={"color":"red"}
    )

    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel(y_col)

    plt.tight_layout()
    plt.show()


ShowTitleBox("Análisis de Regresión - Regresión Plot", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)

ShowRegressionPlot(
    dfTrainOriginal,
    "Daily_Usage_Time (minutes)",
    "Likes_Received_Per_Day",
    "Uso diario vs Likes recibidos"
)

ShowRegressionPlot(
    dfTrainOriginal,
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Uso diario vs Posts por día"
)

ShowRegressionPlot(
    dfTrainOriginal,
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Posts por día vs Likes recibidos"
)

# 5️⃣ Análisis de Clustering

In [ ]:
from sklearn.cluster import KMeans
import sklearn.metrics
from kneed import KneeLocator

# Crear DatasetSplit para clustering
clusterSplit = DatasetSplit(
    Train=Dataset(
        X=dfTrainForClustering,
        y=pandas.DataFrame()
    ),
    Test=Dataset(
        X=dfTestForClustering,
        y=pandas.DataFrame()
    )
)

# Escalar los datos
scaledClusterSplit = ScaleDatasetSplit(
    clusterSplit,
    scaler=CreateScaler(ScalerType.STANDARD)
)


# Función para resolver el mejor k a usar en Kmeans y HC.
def _ResolveBestK(
    bestKSil: Optional[int],
    bestKCh: Optional[int],
    bestKDb: Optional[int],
    bestKElbow: Optional[int],
    fallbackK: int = 2
) -> int:
    for k in (bestKSil, bestKCh, bestKDb, bestKElbow):
        if k is not None:
            return k
    validKs = [
        k for k in (bestKSil, bestKCh, bestKDb, bestKElbow)
        if k is not None
    ]
    if validKs:
        return int(np.median(validKs))
    return fallbackK

# Almacenar los valores de las métricas.
@dataclass(frozen=True)
class MetricData:
    BestK: Optional[int]
    Data: pandas.DataFrame

# Resultados de la evaluación de los mejores k en Kmeans.
@dataclass(frozen=True)
class EvaluateKmeansBestKsResult:
    Elbow: MetricData
    Silhouette: MetricData
    CalinskiHarabasz: MetricData
    DaviesBouldin: MetricData
    ResolvedBestK: int

def _PlotBestKMetrics(
    metrics: list[tuple[str, pandas.DataFrame, str, int | None]],
    title: str,
    figsize: tuple = (12, 8)
):

    n = len(metrics)
    rows = 2
    cols = int(np.ceil(n / 2))

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()

    for ax, (name, df, scoreCol, bestK) in zip(axes, metrics):
        if df.empty:
            ax.set_title(f"{name} (no data)")
            ax.axis("off")
            continue

        ax.plot(df["K"], df[scoreCol], marker="o")
        ax.set_title(name)
        ax.set_xlabel("K")
        ax.set_ylabel(scoreCol)
        ax.grid(alpha=0.3)

        if bestK is not None:
            bestScore = df.loc[df["K"] == bestK, scoreCol].values
            if len(bestScore) > 0:
                ax.scatter(bestK, bestScore[0], color="red", zorder=5)
                ax.axvline(bestK, linestyle="--", color="red", alpha=0.5)
                ax.text(
                    bestK,
                    bestScore[0],
                    f" K={bestK}",
                    color="red",
                    fontsize=9,
                    verticalalignment="bottom"
                )

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# Función para obtener el mejor valor para k en Kmeans.
def EvaluateKmeansBestKs(
    dfScaledAndEncoded: pandas.DataFrame,
    kMax=10,
    randomState=42,
    showPlot: bool = False
) -> EvaluateKmeansBestKsResult:
    silhouetteScores = []
    calinskiHarabaszScores = []
    daviesBouldinScores = []
    inertias = []
    for k in range(1, kMax + 1):
        model = KMeans(n_clusters=k, random_state=randomState)
        labels = model.fit_predict(dfScaledAndEncoded)
        if k > 1:
            silScore = sklearn.metrics.silhouette_score(dfScaledAndEncoded, labels)
            chScore = sklearn.metrics.calinski_harabasz_score(dfScaledAndEncoded, labels)
            dbScore = sklearn.metrics.davies_bouldin_score(dfScaledAndEncoded, labels)
            silhouetteScores.append({"K": k, "Score": silScore})
            calinskiHarabaszScores.append({"K": k, "Score": chScore})
            daviesBouldinScores.append({"K": k, "Score": dbScore})
        inertias.append({"K": k, "Inertia": model.inertia_})

    silhouette_df = pandas.DataFrame(silhouetteScores)
    calinski_df = pandas.DataFrame(calinskiHarabaszScores)
    davies_df = pandas.DataFrame(daviesBouldinScores)
    inertias_df = pandas.DataFrame(inertias)

    bestKSil = int(silhouette_df.sort_values(["Score"], ascending=[False]).iloc[0]["K"])
    bestKCh = int(calinski_df.sort_values(["Score"], ascending=[False]).iloc[0]["K"])
    bestKDb = int(davies_df.sort_values(["Score"], ascending=[True]).iloc[0]["K"])

    kneedle = KneeLocator(
        range(1, kMax + 1),
        inertias_df["Inertia"].values,
        curve="convex",
        direction="decreasing",
    )
    bestKElbow = kneedle.knee

    resolvedBestK = _ResolveBestK(
        bestKSil,
        bestKCh,
        bestKDb,
        bestKElbow
    )

    if showPlot:
        _PlotBestKMetrics(
            metrics=[
                ("Inertia (Elbow)", inertias_df, "Inertia", bestKElbow),
                ("Silhouette", silhouette_df, "Score", bestKSil),
                ("Calinski-Harabasz", calinski_df, "Score", bestKCh),
                ("Davies-Bouldin", davies_df, "Score", bestKDb),
            ],
            title="KMeans – Best K Metrics"
        )
    return EvaluateKmeansBestKsResult(
        MetricData(bestKElbow, inertias_df),
        MetricData(bestKSil, silhouette_df),
        MetricData(bestKCh, calinski_df),
        MetricData(bestKDb, davies_df),
        resolvedBestK,
    )

# Función para evaluar KMeans con un k específico y retornar el dataset con la columna de cluster asignada.
def EvaluateKMeans(
    data: Dataset,
    k: int,
    randomState: int = 42,
    clusterColName: str = "KMeansCluster"
) -> Dataset:

    model = KMeans(
        n_clusters=k,
        random_state=randomState
    )

    labels = model.fit_predict(data.X)

    XClustered = data.X.copy()
    XClustered[clusterColName] = labels

    return Dataset(
        X=XClustered,
        y=data.y  
    )


def Plot2DDataset(dataset: Dataset, targetFeatureCol: str = "Target", title: str = "2D Dataset"):
    plt.figure(figsize=(6, 5))
    colors = (
        dataset.X[targetFeatureCol]
        if dataset.y is not None
        else "gray"
    )
    plt.scatter(
        dataset.X.iloc[:, 0],
        dataset.X.iloc[:, 1],
        s=50,
        c=colors,
        cmap="rainbow"
    )

    plt.xlabel(dataset.X.columns[0])
    plt.ylabel(dataset.X.columns[1])
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


RANDOM_STATE = 42

ShowTitleBox("Evaluación de KMeans - Selección del Mejor K - Gráfico y grupos", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)
kmeansBestKResult = EvaluateKmeansBestKs(clusterSplit.Train.X, 20, RANDOM_STATE, showPlot=True)
kmeansBestK = kmeansBestKResult.Elbow.BestK
kmeansFeatureColName = "KMeansCluster"
KMEANS_CLUSTERS = kmeansBestK if  kmeansBestK is not None else 2

kmeansDataset = EvaluateKMeans(clusterSplit.Train, KMEANS_CLUSTERS, RANDOM_STATE, kmeansFeatureColName)
Plot2DDataset(kmeansDataset, kmeansFeatureColName, f"KMeans Clusters with best k (k={KMEANS_CLUSTERS})")



# 6️⃣ Selección de Características

Se eliminó en la sección 3 la columna ID ya que no aporta información útil para usar en los modelos. Contiene información categórica dispersa.

# 7️⃣ Reducción de Dimensionalidad

Para aplicar en los modelos de la siguiente sección se aplicará PCA.

In [ ]:
targetColumn = "Dominant_Emotion"

# Train
XTrain = dfTrainForClassification.drop(columns=[targetColumn])
yTrain = dfTrainForClassification[[targetColumn]]  

# Test
XTest = dfTestForClassification.drop(columns=[targetColumn])
yTest = dfTestForClassification[[targetColumn]]

classificationSplit = DatasetSplit(
    Train=Dataset(
        X=XTrain,
        y=yTrain
    ),
    Test=Dataset(
        X=XTest,
        y=yTest
    )
)
scaledClassificationSplit = ScaleDatasetSplit(
    classificationSplit,
    scaler=CreateScaler(ScalerType.STANDARD)
)

pcaClassificationSplit = ApplyPCA(scaledClassificationSplit, explainedVarianceRatioSum=0.95, randomState=RANDOM_STATE)

ShowTitleBox("Preparación de datos para Clasificación aplicando PCA", boxLineStyle=TitleBoxLineStyle.BLOCK, color=ConsoleColor.MAGENTA)
ShowDatasetSplitInfo(pcaClassificationSplit, "Train con PCA aplicado")

# 8️⃣ Clasificación Supervisada

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

@dataclass
class DecisionTreeResult:
    Model: DecisionTreeClassifier
    Predictions: pandas.DataFrame
    Accuracy: float
    Precision: float
    Recall: float
    F1: float
    ConfusionMatrix: np.ndarray
    Report: str
    Target: str

def ApplyDecisionTree(
    split: SplitLike,
    targetColumn: str,
    max_depth: int | None = None,
    criterion: Literal["gini", "entropy", "log_loss"] = "gini",
    randomState: int = 42
) -> DecisionTreeResult:

    XTrain = split.Train.X
    yTrain = split.Train.y[targetColumn]
    XTest = split.Test.X
    yTest = split.Test.y[targetColumn]

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        criterion=criterion,
        random_state=randomState
    )

    model.fit(XTrain, yTrain)

    yPred = model.predict(XTest)
    yProba = model.predict_proba(XTest)

    dfPred = pandas.DataFrame(
        {"yReal": yTest.values, "yPred": yPred},
        index=XTest.index
    )

    dfProba = pandas.DataFrame(
        yProba,
        columns=[f"Class-{cls}-Prob" for cls in model.classes_],
        index=XTest.index
    )

    dfPred = pandas.concat([dfPred, dfProba], axis=1)

    return DecisionTreeResult(
        Model=model,
        Predictions=dfPred,
        Accuracy=float(accuracy_score(yTest, yPred)),
        Precision=float(precision_score(yTest, yPred, average="weighted", zero_division=0)),
        Recall=float(recall_score(yTest, yPred, average="weighted", zero_division=0)),
        F1=float(f1_score(yTest, yPred, average="weighted", zero_division=0)),
        ConfusionMatrix=confusion_matrix(yTest, yPred),
        Report=str(classification_report(yTest, yPred)),
        Target=targetColumn
    )


# Para almacenar los resultados de la regresión logística
@dataclass
class LogisticRegressionResult:
    Model: LogisticRegression
    Predictions: pandas.DataFrame
    Accuracy: float
    Precision: float
    Recall: float
    F1: float
    ConfusionMatrix: np.ndarray
    Iters: list[int] | np.ndarray
    Report: str
    Target: str

# Aplica regresión logística al split escalado y retorna el resultado.
def ApplyLogisticRegression(
    split: SplitLike,
    targetColumn: str,
    randomState: int = 42,
    maxIter: int = 200,
    C: float = 1.0,
    penalty: Literal['l1', 'l2', 'elasticnet'] | None = "l2",
    solver: Literal['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'] = "lbfgs"
) -> LogisticRegressionResult:
    
    XTrain = split.Train.X
    yTrain = split.Train.y[targetColumn]
    XTest = split.Test.X
    yTest = split.Test.y[targetColumn]

    model = LogisticRegression(
        max_iter=maxIter,
        random_state=randomState,
        C=C,
        penalty=penalty,
        solver=solver
    )

    model.fit(XTrain, yTrain)

    yPredTrain = model.predict(XTrain)
    yPredTest = model.predict(XTest)

    yProbaTrain = model.predict_proba(XTrain)
    yProbaTest = model.predict_proba(XTest)

    dfProbaTest = pandas.DataFrame(
        yProbaTest,
        index=XTest.index,
        columns=[f"Class-{cls}-Prob" for cls in model.classes_]
    )

    dfPredTest = pandas.DataFrame(
        {
            "yReal": yTest.values,
            "yPred": yPredTest
        },
        index=XTest.index
    )

    dfPredTest = pandas.concat([dfPredTest, dfProbaTest], axis=1)

    acc = accuracy_score(yTest, yPredTest)
    prec = precision_score(yTest, yPredTest, average="weighted", zero_division=0)
    rec = recall_score(yTest, yPredTest, average="weighted", zero_division=0)
    f1 = f1_score(yTest, yPredTest, average="weighted", zero_division=0)
    cm = confusion_matrix(yTest, yPredTest)
    report = classification_report(yTest, yPredTest)

    return LogisticRegressionResult(
        Model=model,
        Predictions=dfPredTest,
        Accuracy=float(acc),
        ConfusionMatrix=cm,
        Precision=float(prec),
        Recall=float(rec),
        F1=float(f1),
        Iters=model.n_iter_,
        Report=str(report),
        Target=targetColumn
    )

@dataclass
class SVMResult:
    Model: SVC
    Predictions: pandas.DataFrame
    Accuracy: float
    Precision: float
    Recall: float
    F1: float
    ConfusionMatrix: np.ndarray
    Report: str
    Target: str

# Aplica SVM al split escalado o con PCA aplicado y retorna el resultado.
def ApplySVM(
    split: SplitLike,
    targetColumn: str,
    C: float = 1.0,
    kernel: Literal["linear", "poly", "rbf", "sigmoid"] = "rbf",
    gamma: Literal["scale", "auto"] | float = "scale",
    degree: int = 3,
    probability: bool = True,
    randomState: int = 42
) -> SVMResult:

    XTrain = split.Train.X
    yTrain = split.Train.y[targetColumn]
    XTest = split.Test.X
    yTest = split.Test.y[targetColumn]

    model = SVC(
        C=C,
        kernel=kernel,
        gamma=gamma,
        degree=degree,
        probability=probability,
        random_state=randomState
    )

    model.fit(XTrain, yTrain)

    yPred = model.predict(XTest)

    dfPred = pandas.DataFrame(
        {"yReal": yTest.values, "yPred": yPred},
        index=XTest.index
    )

    if probability:
        yProba = model.predict_proba(XTest)

        dfProba = pandas.DataFrame(
            yProba,
            columns=[f"Class-{cls}-Prob" for cls in model.classes_],
            index=XTest.index
        )

        dfPred = pandas.concat([dfPred, dfProba], axis=1)

    return SVMResult(
        Model=model,
        Predictions=dfPred,
        Accuracy=float(accuracy_score(yTest, yPred)),
        Precision=float(precision_score(yTest, yPred, average="weighted", zero_division=0)),
        Recall=float(recall_score(yTest, yPred, average="weighted", zero_division=0)),
        F1=float(f1_score(yTest, yPred, average="weighted", zero_division=0)),
        ConfusionMatrix=confusion_matrix(yTest, yPred),
        Report=str(classification_report(yTest, yPred)),
        Target=targetColumn
    )


def GenerateMetricsTable(
    split: SplitLike,
    targetColumn: str
) -> pandas.DataFrame:

    results = []

    lr = ApplyLogisticRegression(split, targetColumn)
    dt = ApplyDecisionTree(split, targetColumn)
    svm = ApplySVM(split, targetColumn)

    results.append({
        "Model": "Logistic Regression",
        "Accuracy": lr.Accuracy,
        "Precision": lr.Precision,
        "Recall": lr.Recall,
        "F1": lr.F1
    })

    results.append({
        "Model": "Decision Tree",
        "Accuracy": dt.Accuracy,
        "Precision": dt.Precision,
        "Recall": dt.Recall,
        "F1": dt.F1
    })

    results.append({
        "Model": "SVM",
        "Accuracy": svm.Accuracy,
        "Precision": svm.Precision,
        "Recall": svm.Recall,
        "F1": svm.F1
    })

    return pandas.DataFrame(results)


dfMetrics = GenerateMetricsTable(pcaClassificationSplit, targetColumn)
display(dfMetrics)

# 9️⃣ Gráficos de rersultados

In [ ]:
def PlotMetricsBarCharts(dfMetrics: pandas.DataFrame):

    metrics = ["Accuracy", "Precision", "Recall", "F1"]
    models = dfMetrics["Model"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        axes[i].bar(models, dfMetrics[metric])
        axes[i].set_title(metric)
        axes[i].set_ylim(0, 1)
        axes[i].set_ylabel("Score")
        axes[i].set_xlabel("Model")
        axes[i].grid(axis="y", linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()

def PlotModelsConfusionMatrices(
    split: SplitLike,
    targetColumn: str
):

    lr = ApplyLogisticRegression(split, targetColumn)
    dt = ApplyDecisionTree(split, targetColumn)
    svm = ApplySVM(split, targetColumn)

    classNames = sorted(split.Train.y[targetColumn].unique())
    print("ℹ️ Logistic Regression")
    PlotConfusionMatrix(
        lr.ConfusionMatrix,
        title="Logistic Regression - Confusion Matrix",
        classNames=classNames
    )

    print("ℹ️ Decision Tree")
    PlotConfusionMatrix(
        dt.ConfusionMatrix,
        title="Decision Tree - Confusion Matrix",
        classNames=classNames
    )

    print("ℹ️ SVM")
    PlotConfusionMatrix(
        svm.ConfusionMatrix,
        title="SVM - Confusion Matrix",
        classNames=classNames
    )

PlotMetricsBarCharts(dfMetrics)
PlotModelsConfusionMatrices(pcaClassificationSplit, targetColumn)

# 